In [ ]:

import torch

from src.data.dataset.geometric_wrapper import TensorGeometricModelWrapper

from its.search import InverseTransformationSearch
from search.gradient_descent import ParallelGradientDescent

from src.utils.sampling import BatchNegativeSampler

#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_mnist"

default_architecutre_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "coil100": "coil_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:
import torch

print(torch.__version__)

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info, path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

train_loader_augmented = dataset_dict.get('train_loader_augmented', None)

In [ ]:
x = next(iter(test_loader_transformed))[0]

batch_size = next(iter(train_loader))[0].shape[0]

from src.utils.eval.vis import vis_dataset

vis_dataset(train_loader, val_loader, test_loader_transformed)
from model.train import train_and_get_model, train_or_load_energy_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparison_augmented_models")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train = f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model, model_dir_path, modelname, train_loader, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
dataset_augmented = train_loader_augmented.dataset

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]


In [ ]:
def evaluate_and_store_results(model, modelname, test_loader, test_loader_transformed, device, savepath):
    #check if stored results exist
    if savepath is not None and os.path.exists(savepath):
        import json
        with open(savepath, "r") as f:
            results = json.load(f)
        print(f"Loaded results for {modelname} from {savepath}:")
        print("Transformed test set results:")
        print(results["transformed_test_results"])
        print("Original test set results:")
        print(results["original_test_results"])
        return results

    res_transformed = evaluate_base_model(model, test_loader_transformed, device)
    res = evaluate_base_model(model, test_loader, device)
    print(f"Results for {modelname}:")
    print("Transformed test set results:")
    print(res_transformed)
    print("Original test set results:")
    print(res)
    if savepath is not None:
        import json
        results = {
            "modelname": modelname,
            "transformed_test_results": res_transformed,
            "original_test_results": res,
        }
        with open(savepath, "w") as f:
            json.dump(results, f, indent=4)

    return res


In [ ]:
result_folder = os.path.join(results_dir_path, transform_name)
os.makedirs(result_folder, exist_ok=True)

In [ ]:
#check main model
res = evaluate_and_store_results(model, modelname, test_loader, test_loader_transformed, device, savepath(modelname))

In [ ]:
class TensorGeometricModelUnwrapper(torch.nn.Module):
    """
    Wrapper for a torch_geometric model that receives a tuple of (pos, y) as input and creates
    a Data object from it that is passed to the model.
    """

    def __init__(self):
        super(TensorGeometricModelUnwrapper, self).__init__()

    def forward(self, data):
        # pos and y are batched tensors from a DataLoader
        # need to reconstruct the original Data objects for torch_geometric models

        pos = data.pos
        batch = data.batch
        #split pos into individual tensors based on batch
        pos_list = torch.split(pos, torch.bincount(batch).tolist())
        return torch.stack(pos_list)

In [ ]:
model_augmented = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname_augmented = f"{dataset}_{architecture}_augmented"

train_and_get_model(model_augmented, model_dir_path, modelname_augmented, train_loader_augmented,
                    val_loader_transformed, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
    }, load_if_exists=True)

print("Augmented model results:")
res = evaluate_and_store_results(model_augmented, modelname_augmented, test_loader, test_loader_transformed, device,
                                 savepath(modelname_augmented))

del model_augmented
torch.cuda.empty_cache()


In [ ]:
model_augmented = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname_augmented = f"{dataset}_{architecture}_augmented_val"

train_and_get_model(model_augmented, model_dir_path, modelname_augmented, train_loader_augmented, val_loader,
                    trainer_kwargs={
                        "accelerator": "auto",
                        "max_epochs": dataset_info.epochs,
                        "precision": "16-mixed",
                    }, load_if_exists=True)

res = evaluate_and_store_results(model_augmented, modelname_augmented, test_loader, test_loader_transformed, device,
                                 savepath(modelname_augmented))
torch.cuda.empty_cache()


In [ ]:
if dataset == "modelnet10":
    architecture2 = "pointnetplus_euclidean"
    model_unaugmented_euclidean = get_network(dataset_info, architecture2, num_classes=n_classes).to(device)

    modelname_unaugmented_euclidean = f"{dataset}_{architecture2}_no_aug"
    train_and_get_model(model_unaugmented_euclidean, model_dir_path, modelname_unaugmented_euclidean, train_loader,
                        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
        }, load_if_exists=True)

    print("Unaugmented Euclidean model results:")
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader_transformed, device)
    print(res)
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader, device)
    print(res)

    model_unaugmented_euclidean = None
    torch.cuda.empty_cache()

    model_augmented_euclidean = get_network(dataset_info, architecture2, num_classes=n_classes).to(device)
    modelname_augmented_euclidean = f"{dataset}_{architecture2}_augmented"
    train_and_get_model(model_augmented_euclidean, model_dir_path, modelname_augmented_euclidean,
                        train_loader_augmented, val_loader_transformed, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
        }, load_if_exists=True)

    evaluate_and_store_results(model_augmented_euclidean, modelname_augmented_euclidean, test_loader,
                               test_loader_transformed, device, savepath(modelname_augmented_euclidean))


In [ ]:
if dataset == "modelnet10":
    architecture2 = "pointnetplus_euclidean"
    model_unaugmented_euclidean = get_network(dataset_info, architecture2, num_classes=n_classes).to(device)

    modelname_unaugmented_euclidean = f"{dataset}_{architecture2}_no_aug"
    train_and_get_model(model_unaugmented_euclidean, model_dir_path, modelname_unaugmented_euclidean, train_loader,
                        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
        }, load_if_exists=True)

    print("Unaugmented Euclidean model results:")
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader_transformed, device)
    print(res)
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader, device)
    print(res)

    model_unaugmented_euclidean = None
    torch.cuda.empty_cache()



In [ ]:
if is_image_data:
    import torch, gc, time

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{modelname} total parameters: {total_params:,}")
    print(f"{modelname} trainable parameters: {trainable_params:,}")

    import torchinfo

    print(torchinfo.summary(
        model,
        input_size=(dataset_info.batch_size, *dataset_info.input_size),
        col_names=("output_size", "num_params", "mult_adds"),
        depth=3
    ))

    model.eval()
    dummy_input = torch.randn(dataset_info.batch_size, *dataset_info.input_size).to(device)
    _ = model(dummy_input)
    torch.cuda.synchronize()

    torch.cuda.synchronize()
    start_time = time.time()
    with torch.no_grad():
        for _ in range(2000):
            _ = model(dummy_input)
    torch.cuda.synchronize()
    elapsed = (time.time() - start_time) / 2000
    print(f"{modelname} avg forward pass time: {elapsed * 1000:.2f} ms")

In [ ]:
from model.basic_networks import convert_flexible_resnet_to_escnn

if is_image_data:
    model_escnn = convert_flexible_resnet_to_escnn(model).to(device)

    import time

    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(100):
        outputs = model(x)
        torch.cuda.synchronize()
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 100:.6f} seconds")

    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(100):
        outputs = model(x)
        torch.cuda.synchronize()
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 100:.6f} seconds")

In [ ]:
if is_image_data:
    import time

    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(100):
        outputs = model_escnn(x)
        torch.cuda.synchronize()
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 100:.6f} seconds")

    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(100):
        outputs = model_escnn(x)
        torch.cuda.synchronize()
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 100:.6f} seconds")

In [ ]:
import torch
import torch.nn.functional as F


def rotate_tensor_90(x, k=1):
    # Rotates tensor by 90 degrees k times along spatial dimensions
    # x shape: (B, C, H, W)
    return torch.rot90(x, k=k, dims=[-2, -1])


def test_rotation_invariance(model, x, device):
    model.eval()
    x = x.to(device)

    with torch.no_grad():
        # Original logits
        logits_orig = model(x)

        # Rotate input by 90°
        x_rot = rotate_tensor_90(x, k=1)
        logits_rot = model(x_rot)

        # Compare logits directly
        error = F.mse_loss(logits_orig, logits_rot)

    return error.item()


if is_image_data:
    # --- Run the test ---
    x, y = next(iter(test_loader))
    x = x.to(device)
    invariance_error = test_rotation_invariance(model_augmented, x, device)
    print(f"Rotation invariance MSE error (classification logits): {invariance_error:.8f}")

    invariance_error = test_rotation_invariance(model_escnn, x, device)
    print(f"Rotation invariance MSE error (classification logits): {invariance_error:.8f}")

In [ ]:
dataset_info

In [ ]:
from src.utils.replacer import replace_scaling_transforms, replace_rotation_transforms
import torch
import torch.nn as nn


class SpatialTransformer(nn.Module):
    """
    A  STN wrapper that predicts a transformation and applies it
    before passing the result to a main classifier.
    """

    def __init__(
            self,
            main_model,
            localization_net,
            localization_dim,
            transformation_problem,
            pretransform=None,
            clamp=False
    ):
        super().__init__()
        self.main_model = main_model
        self.localization_net = localization_net
        self.transformation_problem = transformation_problem
        self.pretransform = pretransform
        self.clamp = clamp

        # The regression head: maps features to transformation parameters
        reg_dim = transformation_problem.calc_complete_size()
        self.regression_net = nn.Linear(localization_dim, reg_dim)

        self.reset_parameters()

    def reset_parameters(self):
        """Initializes the regressor to output an identity transformation."""
        nn.init.normal_(self.regression_net.weight, 0, 0.01)

        with torch.no_grad():
            identity_params = self.transformation_problem.get_identity_parameters()
            # Ensure bias starts the network at (or near) identity
            self.regression_net.bias.copy_(identity_params.flatten())

    def forward(self, x):
        if self.pretransform is not None:
            with torch.no_grad():
                x = self.pretransform(x)

        # Localization: Extract spatial features
        loc_features = self.localization_net(x)
        loc_features = loc_features.flatten(1)

        # Regression: Predict transformation parameters
        params = self.regression_net(loc_features)

        # Warp the input image
        x_transformed = self.transformation_problem.transform(x, params)

        # 5. Classify: Pass the 'corrected' image to the main model
        return self.main_model(x_transformed)

    def get_transformation_params(self, x):
        """Helper to get transformed images"""
        loc_features = self.localization_net(x).flatten(1)
        return self.regression_net(loc_features)


architecture_quarter = architecture + "_quarter"
localization_dim = 128
localization_net = get_network(
    dataset_info,
    architecture_quarter,
    num_classes=localization_dim
).to(device)
base = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
# 1. Instantiate the Spatial Transformer wrapper
# Note: Ensure you have a 'transformation_problem' defined (e.g., AffineTransformation)
from src.utils.transformation_problem import TransformationProblem

from data.transformation import get_transformation_sequence_images

transform_seq2 = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

from src.utils.replacer import replace_rotation_transforms, replace_rotation_transforms_2vec

if dataset == "modelnet10":
    transform_seq2 = replace_rotation_transforms_2vec(transform_seq2)
else:
    transform_seq2 = replace_rotation_transforms(transform_seq2)
if dataset == "coil100":
    transform_seq2 = replace_scaling_transforms(transform_seq2, replace_scale=False)
transform_seq_spatial = transform_seq2

transformation_problem = TransformationProblem(None, transform_seq_spatial)

model_spatial = SpatialTransformer(
    main_model=base,
    localization_net=localization_net,
    localization_dim=localization_dim,
    transformation_problem=transformation_problem,
).to(device)

modelname_spatial = f"{dataset}_{architecture}_spatial_transformer"

# 2. Train the STN model
train_and_get_model(
    model_spatial,
    model_dir_path,
    modelname_spatial,
    train_loader_augmented,
    val_loader_transformed,
    trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32" if dataset == "modelnet10" or dataset == "tu_berlin" else "16-mixed",
    },
    load_if_exists=True
)

# 3. Evaluate and store results
print("Spatial Transformer model results:")
res_spatial = evaluate_and_store_results(
    model_spatial,
    modelname_spatial,
    test_loader,
    test_loader_transformed,
    device,
    savepath(modelname_spatial)
)

del model_spatial
torch.cuda.empty_cache()

In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScaleEuclidean
from types import SimpleNamespace
from external.vnn.models.vn_dgcnn_cls import get_model


class GeometricToTensorUnwrapper(nn.Module):
    """
    Extracts 'pos' from a torch_geometric.Batch and reshapes it
    for models expecting [Batch, Channels, Num_Points].
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, data):
        batch_size = data.num_graphs
        num_nodes = data.pos.shape[0] // batch_size

        # Reshape to [B, N, 3]
        x = data.pos.view(batch_size, num_nodes, 3)

        return self.model(x)


def _build_vn_dgcnn(num_classes=10, normalize=True, n_knn=20, pooling="mean"):
    #Initialize the raw VN-DGCNN
    args = SimpleNamespace()
    args.n_knn = n_knn
    args.pooling = pooling
    vn_core = get_model(args, num_classes, normal_channel=False)

    unwrapped_core = GeometricToTensorUnwrapper(vn_core)

    if normalize:
        pipeline = nn.Sequential(BatchNormalizeScaleEuclidean(), unwrapped_core)
    else:
        pipeline = unwrapped_core

    model = TensorGeometricModelWrapper(pipeline)

    return model

In [ ]:
if dataset == "modelnet10":
    model_vn = _build_vn_dgcnn(
        num_classes=n_classes,
        normalize=True
    ).to(device)

    modelname_vn = f"{dataset}_{architecture}_vn_dgcnn"


    train_and_get_model(
        model_vn,
        model_dir_path,
        modelname_vn,
        train_loader,
        val_loader,
        trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32" if dataset == "modelnet10" else "16-mixed",
        },
        load_if_exists=True
    )

    print("VN-DGCNN model results:")
    res_vn = evaluate_and_store_results(
        model_vn,
        modelname_vn,
        test_loader,
        test_loader_transformed,
        device=device,
        savepath=savepath(modelname_vn)
    )

In [ ]:
import torch

torch.cuda.get_device_name(0)
torch.cuda.get_device_capability(0)

In [ ]:
transformation_problem_sample = TransformationProblem(None, get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device))


In [ ]:
class TestTimeAugmenter(nn.Module):
    """
    Wraps a model to perform Test-Time Augmentation (TTA).
    No training required.
    """

    def __init__(self, model, transformation_problem, n_samples=8):
        super().__init__()
        self.model = model
        self.transformation_problem = transformation_problem
        self.n_samples = n_samples

    def forward(self, x):
        # If we are in training mode, just pass through (TTA is for inference)
        if self.training:
            return self.model(x)

        batch_size = x.shape[0]
        # Store accumulated logits
        combined_logits = None

        # 1. Original image prediction
        combined_logits = self.model(x)

        # 2. Sample random transformations and average results
        for _ in range(self.n_samples - 1):
            random_params = self.transformation_problem.initial_param(batch_size).to(x.device)

            x_aug = self.transformation_problem.transform(x, random_params)
            logits = self.model(x_aug)

            combined_logits += logits

        # Return the mean logits
        return combined_logits / self.n_samples

In [ ]:

model_tta = TestTimeAugmenter(
    model=model,
    transformation_problem=transformation_problem,
    n_samples=8
).to(device)

modelname_tta = f"{dataset}_{architecture}_tta"

print("TTA Model results (Inference only):")
res_tta = evaluate_and_store_results(
    model_tta,
    modelname_tta,
    test_loader,
    test_loader_transformed,
    device,
    savepath(modelname_tta)
)

del model_tta
torch.cuda.empty_cache()

In [ ]:

model_tta = TestTimeAugmenter(
    model=model,
    transformation_problem=transformation_problem,
    n_samples=60
).to(device)

modelname_tta = f"{dataset}_{architecture}_tta_large"

print("TTA Model results (Inference only):")
res_tta = evaluate_and_store_results(
    model_tta,
    modelname_tta,
    test_loader,
    test_loader_transformed,
    device,
    savepath(modelname_tta)
)

del model_tta
torch.cuda.empty_cache()

In [ ]:

model_tta = TestTimeAugmenter(
    model=model_augmented,
    transformation_problem=transformation_problem,
    n_samples=8
).to(device)

modelname_tta = f"{dataset}_{architecture}_augmented_tta"

print("TTA Model results (Inference only):")
res_tta = evaluate_and_store_results(
    model_tta,
    modelname_tta,
    test_loader,
    test_loader_transformed,
    device,
    savepath(modelname_tta)
)

del model_tta
torch.cuda.empty_cache()

In [ ]:

model_tta = TestTimeAugmenter(
    model=model_augmented,
    transformation_problem=transformation_problem,
    n_samples=60
).to(device)

modelname_tta = f"{dataset}_{architecture}_augmented_tta_large"

print("TTA Model results (Inference only):")
res_tta = evaluate_and_store_results(
    model_tta,
    modelname_tta,
    test_loader,
    test_loader_transformed,
    device,
    savepath(modelname_tta)
)

del model_tta
torch.cuda.empty_cache()

In [ ]:
import pytorch_lightning as pl

if is_image_data:
    configs = [
        ("escnn", lambda m: convert_flexible_resnet_to_escnn(m)),
        #("escnn_48", lambda m: convert_flexible_resnet_to_escnn(m,rotations=48)),
    ]
    epoch_config = {
        "escnn": dataset_info.epochs,
        "escnn_48": dataset_info.epochs // 5,
    }
    preccision_config = {"escnn": "16-mixed", "escnn_48": "16-mixed"}

    # Speed test and parameter count of main model
    for suffix, converter in configs:
        print(f"\n==== Training {suffix.upper()} model ====")
        modelname = f"{dataset}_{architecture}_{suffix}"
        model_tmp = converter(model).to(device)

        # # --- Parameter count ---
        total_params = sum(p.numel() for p in model_tmp.parameters())
        trainable_params = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
        print(f"{modelname} total parameters: {total_params:,}")
        print(f"{modelname} trainable parameters: {trainable_params:,}")

        print(model_tmp)

        import torch
        import gc
        import pytorch_lightning as pl  # <--- ADD THIS LINE (if it's missing)


        class gc_callback(pl.callbacks.Callback):
            def on_validation_end(self, trainer, pl_module):
                gc.collect()
                torch.cuda.empty_cache()


        # --- Train model ---
        train_and_get_model(
            model_tmp,
            model_dir_path,
            modelname,
            train_loader_augmented,
            val_loader_transformed,
            trainer_kwargs={
                "accelerator": "auto",
                "max_epochs": epoch_config[suffix],
                "precision": preccision_config[suffix],
            },
            load_if_exists=True, strict=False,
        )

        evaluate_and_store_results(model_tmp, modelname, test_loader, test_loader_transformed, device,
                                   savepath(modelname))
        del model_tmp
        gc.collect()
        torch.cuda.empty_cache()



In [ ]:

W = 10

In [ ]:
def _visualize_strokes(data, labels, title, n_rows, n_cols, save_path):
    """Helper to visualize stroke sequence data."""
    n_samples = min(len(data), n_rows * n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(W * 4, n_rows * W * 4 / n_cols))

    for i in range(n_samples):
        row, col = i // n_cols, i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]

        seq = data[i].cpu().numpy()
        mask = seq[:, 3] == 1
        if np.any(mask):
            dx, dy, pen = seq[mask, 0], seq[mask, 1], seq[mask, 2]
            x, y = np.cumsum(dx), np.cumsum(dy)

            xs, ys = [], []
            for xi, yi, p in zip(x, y, pen):
                xs.append(xi)
                ys.append(-yi)
                if p == 1:
                    ax.plot(xs, ys, 'k-', linewidth=1.5)
                    xs, ys = [], []
            if xs:
                ax.plot(xs, ys, 'k-', linewidth=1.5)

        ax.axis('equal')
        ax.axis('off')

    # Hide extra subplots
    for i in range(n_samples, n_rows * n_cols):
        row, col = i // n_cols, i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.axis('off')

    if title:
        fig.suptitle(title)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()

In [ ]:



from src.utils.transforms.apply import grid_resample
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

from src.utils.replacer import replace_rotation_transforms_2vec

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
print(transform_seq.transformations)

In [ ]:

from model.basic_networks import make_deterministic

make_deterministic(model)



In [ ]:
from get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture, transform_name=None, dataset_info=dataset_info)
train_cache = create_layer_embedding_cache(model, train_loader_no_shuffle, cache_config, embedding_cache_path,
                                           device=device)


In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
detectors = ["knn", "per_class_knn", "knn_mixed", "per_class_knn_mixed", "knn_mixed_faiss", "knn_itf", "vim", "react",
             "dice", "ash", "she", "laplace_mi", "laplace_energy", "laplace_weighted", "trust_score", "openmax",
             "mahalanobis", "rmd", "class_prototype", "react_all", "energy", "per_class_prototype",
             "single_mahalanobis", "single_rmd", "single_mahalanobis_individual", "single_rmd_individual",
             "mahalanobis_individual", "rmd_individual", "prototype_multi", "laplace_entropy", "adjusted_entropy",
             "nn_guided", "nn_guided_one"]

In [ ]:

import os
import json
import math
import torch
from typing import Tuple, Optional, Any, Dict

from hyper_param.ood.base_prepare import create_ood_problem, get_default_ood_params, find_best_detector_and_instantiate

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)

best_detector, best_problem, best_score, second_choice, second_problem, second_score = find_best_detector_and_instantiate(
    base_results_dir=base_results_dir,
    detectors=detectors,
    model=model,
    train_cache=train_cache,
    transform_seq_arg=transform_seq,
    dataset_info=dataset_info,
    architecture=architecture,
    device=device,
    val_id_loader=val_loader_transformed,
    val_ood_loader=val_loader_transformed,
)
#apply best problem to test loader
save_path_best = os.path.join(
    base_results_dir,
    best_detector,
    "test_results.json"
)
from search.random_search import RSLR

optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3, local_opt_kwargs={"lr": 0.1})
if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=best_problem,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, save_path=save_path_best, repeats=5)

In [ ]:
from src.utils.eval.ood_performance import ITSWRAPPER
import its

its_opt = None
extend = 0
if dataset == "tu_berlin":
    its_opt = ITSWRAPPER(
        its.search.InverseTransformationSearch(model, None, None, n_hypotheses=1, n_samples=12, extend=extend,
                                               gaussian_filter_channel_wise=False))
elif dataset == "modelnet10":
    its_opt = ITSWRAPPER(
        its.search.InverseTransformationSearch(model, None, None, n_hypotheses=4, n_samples=6, extend=extend,
                                               gaussian_filter_channel_wise=True))
elif dataset == "coil100":
    its_opt = ITSWRAPPER(
        its.search.InverseTransformationSearch(model, None, None, n_hypotheses=1, n_samples=10, extend=extend,
                                               gaussian_filter_channel_wise=True))
else:
    its_opt = ITSWRAPPER(
        its.search.InverseTransformationSearch(model, None, None, n_hypotheses=2, n_samples=6, extend=extend,
                                               gaussian_filter_channel_wise=False))


In [ ]:
from src.utils.transformation_problem import TransformationProblem
from confidence.direct.logit_based import EnergyConfidence
from confidence.model.single_pass import SinglePassConfidence

# 1. Setup shared sequence
transform_seq_its = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

# Shared config to avoid repetition
problem_kwargs = {
    "transform_sequence": transform_seq_its,
    "consolidate_method": "consolidate_simple",
    "max_batch_size": dataset_info.batch_size_search
}

# 2. Define confidence modules
logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)

# 3. Instantiate problems
# Ensure 'best_problem' is defined somewhere above this!
its_problem = TransformationProblem(None, **problem_kwargs)
its_problem2 = TransformationProblem(logit_energy, **problem_kwargs)
best_its_problem = TransformationProblem(best_problem.confidence_module, **problem_kwargs)

# 4. Prepare problems
its_problem = ITSWRAPPER._prepare_problem(its_problem)
its_problem2 = ITSWRAPPER._prepare_problem(its_problem2)
best_its_problem = ITSWRAPPER._prepare_problem(best_its_problem)

In [ ]:
transform_seq_its.transformations

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=its_problem,
    test_loader=test_loader_transformed,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_transformed333"),
    show_progress=True,
    repeats=1
)

In [ ]:
res_its_transformed_energy = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=its_problem2,
    test_loader=test_loader_transformed,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_transformed_energy333"),
    show_progress=True,
    repeats=1
)

In [ ]:
# Evaluate ITS on transformed test set
res_its_transformed = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=its_problem,
    test_loader=test_loader_transformed,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_transformed"),
    show_progress=True,
    repeats=1
)

# Evaluate ITS on original test set
res_its_original = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=its_problem,
    test_loader=test_loader,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_original"),
    show_progress=True,
    repeats=1
)

# Consolidate results for evaluate_and_store_results format if needed
results_its = {
    "modelname": "ITS",
    "transformed_test_results": res_its_transformed,
    "original_test_results": res_its_original,
}
print(results_its)
with open(savepath("results_its"), "w") as f:
    import json

    json.dump(results_its, f, indent=4)


In [ ]:
# Evaluate ITS on transformed test set
res_its_transformed_energy = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=its_problem2,
    test_loader=test_loader_transformed,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_transformed_energy"),
    show_progress=True,
    repeats=1
)

# Evaluate ITS on original test set
res_its_original_energy = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=its_problem2,
    test_loader=test_loader,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_original_energy"),
    show_progress=True,
    repeats=1
)

# Consolidate results for evaluate_and_store_results format if needed
results_its_energy = {
    "modelname": "ITS",
    "transformed_test_results": res_its_transformed_energy,
    "original_test_results": res_its_original_energy,
}
print(results_its_energy)
with open(savepath("results_its_energy"), "w") as f:
    import json

    json.dump(results_its_energy, f, indent=4)


In [ ]:
# Evaluate ITS on transformed test set
res_its_transformed_best = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=best_its_problem,
    test_loader=test_loader_transformed,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_transformed_best_detector"),
    show_progress=True,
    repeats=1
)

# Evaluate ITS on original test set
res_its_original_best = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=its_opt,
    problem=best_its_problem,
    test_loader=test_loader,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("results_its_original_best_detector"),
    show_progress=True,
    repeats=1
)

# Consolidate results for evaluate_and_store_results format if needed
results_its_best = {
    "modelname": "ITS",
    "transformed_test_results": res_its_transformed_best,
    "original_test_results": res_its_original_best,
}
print(results_its_best)
with open(savepath("results_its_best"), "w") as f:
    import json

    json.dump(results_its_best, f, indent=4)


In [ ]:
modelname = f"{dataset}_{architecture}"

In [ ]:
from typing import Optional, Dict, Any, List, Tuple


def _load_score(base_results_dir: str, det_name: str) -> Optional[Dict[str, float]]:
    """Helper to load evaluation scores (mean/se) for a detector from its results file."""
    eval_path = os.path.join(base_results_dir, det_name, "eval_results.json")
    eval_default_path = os.path.join(base_results_dir, det_name, "eval_results_default.json")

    for p in (eval_path, eval_default_path):
        if os.path.exists(p):
            try:
                with open(p, "r") as f:
                    data = json.load(f)
            except Exception:
                continue
            mean = data.get("accuracy_mean")
            se = data.get("accuracy_se", data.get("accuracy_std"))

            # Normalize types
            mean_val = float(mean) if isinstance(mean, (int, float)) else math.nan
            se_val = float(se) if isinstance(se, (int, float)) else math.nan

            if not math.isnan(mean_val):
                return {"mean": mean_val, "se": se_val}
    return None

#TODO maybe change this
def load_best_detector_scores_by_components(
        current_path: str,
        detectors: List[str],
        dataset_info: Any,
        architecture: str,
        dataset_name: str,
        test_filename: str = "test_results.json",
        score_key: str = "accuracy_mean"
) -> Tuple[
    Optional[str],  # Best Detector Name
    Optional[float],  # Final Test Score (from test_filename)
    Optional[float]  # Optimized Selection Score (from _load_score)
]:
    """
    Constructs the base results directory, finds the detector with the best
    recorded 'accuracy_mean' (Optimized Selection Score), and loads its
    Optimized Selection Score and final test score.
    """

    # 1. Reconstruct the base_results_dir path
    transform_seq_name = getattr(dataset_info, "transform_seq_name", "default")

    base_results_dir = os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        str(dataset_name),
        str(architecture),
        transform_seq_name,
    )

    # 2. Find the best detector and its Optimized Selection Score
    best_detector: Optional[str] = None
    best_score_val = -math.inf
    optimized_selection_score: Optional[float] = None  # Renamed variable
    best_se: Optional[float] = None
    for det in detectors:
        score_dict = _load_score(base_results_dir, det)
        if score_dict is None:
            continue
        mean = score_dict["mean"]
        se = score_dict["se"]

        if mean > best_score_val:
            best_score_val = mean
            best_detector = det
            # Store the score used for selection
            optimized_selection_score = mean  # Storing the score
            best_se = se

    if best_detector is None:
        print("No valid scored detector found.")
        return None, None, None

    print(f"\nBest Detector Found: **{best_detector}** (Optimized Selection Score: {optimized_selection_score:.4f})")
    det_dir = os.path.join(base_results_dir, best_detector)

    # 3. Load Test Results and extract the final score
    final_test_score: Optional[float] = None  # Renamed variable
    results_path = os.path.join(det_dir, test_filename)

    if os.path.exists(results_path):
        try:
            with open(results_path, "r") as f:
                test_results = json.load(f)

                # Extract the final test score
                test_score = test_results.get(score_key)
                if isinstance(test_score, (int, float)):
                    final_test_score = float(test_score)
                    print(f"Loaded **Final Test Score** ({score_key}): {final_test_score:.4f}")
                else:
                    print(f"Test results found, but could not extract score key: '{score_key}'")

        except Exception as e:
            print(f"Error loading final test results: {e}")
    else:
        print(f"Final Test results file **not found** at: {results_path}")
    if score_key == "accuracy_mean":
        # Return: name, final_test_score, optimized_selection_score
        return best_detector, final_test_score, optimized_selection_score
    elif score_key == "accuracy_se":
        return best_detector, final_test_score, best_se
    else:
        raise ValueError("score_key must be either 'accuracy_mean' or 'accuracy_se'")

In [ ]:
load_best_detector_scores_by_components(current_path, detectors, dataset_info, architecture, dataset)


In [ ]:
from search.random_search import RSLR

optimizer_120 = RSLR(initial_samples=120 - 27, local_runs=3, local_max_steps=4, local_opt_kwargs={"lr": 0.1})
if dataset == "tu_berlin":
    optimizer_120 = RSLR(initial_samples=120, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})

In [ ]:

from src.utils.augments import small_affine_augment_2d, ComposeAugmentations, random_blur_or_sharpen
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

architecture_half = architecture + "_half"

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None


def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


energy_model2_half = get_network(dataset_info, architecture_half, num_classes=1).to(device)
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_half = train_or_load_energy_model(
    energy_model2_half, model_dir_path, f"{modelname}_energy2_half", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_half.to(device).eval()
problem_energy_half = TransformationProblem(energy_conf_half, transform_seq, consolidate_method="consolidate_simple")


def savepath_later_layer(label: str) -> str:
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                    "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")


res_120 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath_later_layer("learned_energy_confidence_half_transformed_120"), show_progress=True,
    repeats=5)

res_120_real_data = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer("learned_energy_confidence_half_transformed_120_untransformed"),
    repeats=5)

del energy_model2_half
del energy_conf_half
del problem_energy_half

print(res_120["accuracy_mean"])






In [ ]:

from src.utils.augments import small_affine_augment_2d, ComposeAugmentations, random_blur_or_sharpen
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from src.utils.transformation_problem import TransformationProblem

architecture_quarter = architecture + "_quarter"

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None


def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from search.random_search import RSLR

optimizer_240 = RSLR(initial_samples=195, local_runs=5, local_max_steps=4, local_opt_kwargs={"lr": 0.1})
if dataset == "tu_berlin":
    optimizer_240 = RSLR(initial_samples=240, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})
if dataset == "modelnet10":
    optimizer_240 = RSLR(initial_samples=180 - 27, local_runs=3, local_max_steps=4, local_opt_kwargs={"lr": 0.1})

optimizer_240_include_init = RSLR(initial_samples=195, local_runs=5, local_max_steps=4, local_opt_kwargs={"lr": 0.1},
                                  include_zero_always=True)
if dataset == "tu_berlin":
    optimizer_240_include_init = RSLR(initial_samples=240, local_runs=1, local_max_steps=0,
                                      local_opt_kwargs={"lr": 0.1}, include_zero_always=True)
if dataset == "modelnet10":
    optimizer_240_include_init = RSLR(initial_samples=180 - 27, local_runs=3, local_max_steps=4,
                                      local_opt_kwargs={"lr": 0.1}, include_zero_always=True)

energy_model2_quarter = get_network(dataset_info, architecture_quarter, num_classes=1).to(device)
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_quarter = train_or_load_energy_model(
    energy_model2_quarter, model_dir_path, f"{modelname}_energy2_quarter", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_quarter.to(device).eval()
problem_energy_quarter = TransformationProblem(energy_conf_quarter, transform_seq,
                                               consolidate_method="consolidate_simple")


def savepath_later_layer(label: str) -> str:
    model_dir_path = os.path.join(current_path, "experiment_files", "models")
    embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
    # Add results dir and helper for save paths
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                    "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")


name = "learned_energy_confidence_quarter_transformed_240"
name_un = "learned_energy_confidence_quarter_untransformed_240"
name_un_included = "learned_energy_confidence_quarter_untransformed_240_inc"
if dataset == "modelnet10":
    name = "learned_energy_confidence_quarter_transformed_180"
    name_un = "learned_energy_confidence_quarter_untransformed_180"
    name_un_included = "learned_energy_confidence_quarter_untransformed_180_inc"

res_large_transformed = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath_later_layer(name), show_progress=True,
    repeats=5)

res_large_real_data = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer(name_un),
    repeats=5)

res_large_real_data_included = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240_include_init, problem=problem_energy_quarter,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer(name_un_included),
    repeats=5)

print(res_large_real_data)
print(res_large_real_data_included)

del energy_model2_quarter
del energy_conf_quarter
del problem_energy_quarter








In [ ]:

def load_energy_learned(
        dataset_info,
        transform_name,
        current_path,
        dataset,
        architecture
):
    """
    Loads pre-computed evaluation results for the 'learned_energy_confidence_half_transformed_120'
    variants.

    Note: This function assumes that 'load_or_run_evaluate_confidence_and_search'
    supports a 'load_only=True' flag, which is set here to ensure it only loads
    the results from the save_path and does not re-run the evaluation.
    """

    def _savepath_later_layer(label: str) -> str:
        """Utility function to calculate the standardized save path for results."""
        # Add results dir and helper for save paths
        results_dir_path = os.path.join(
            current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods"
        )
        # Ensure the base directory exists before attempting to join with transform_name
        os.makedirs(results_dir_path, exist_ok=True)

        safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
        # Use transform_name as a subdirectory
        return os.path.join(results_dir_path, transform_name, f"{safe}.json")

    # --- 1. Load Transformed Data Result ---
    save_path_transformed = _savepath_later_layer(
        "learned_energy_confidence_quarter_transformed_240"
    )
    if dataset == "modelnet10":
        save_path_transformed = _savepath_later_layer(
            "learned_energy_confidence_quarter_transformed_180"
        )
    try:
        with open(save_path_transformed, "r") as f:
            res_120 = json.load(f)
    except:
        res_120 = None
    # --- 2. Load Untransformed Data Result ---
    save_path_untransformed = _savepath_later_layer(
        "learned_energy_confidence_quarter_untransformed_240"
    )
    if dataset == "modelnet10":
        save_path_untransformed = _savepath_later_layer(
            "learned_energy_confidence_quarter_untransformed_180"
        )
    try:
        with open(save_path_untransformed, "r") as f:
            res_120_real_data = json.load(f)
    except:
        res_120_real_data = None

    return res_120_real_data, res_120

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load all results from the experiment
result_folder = os.path.join(results_dir_path, transform_name)

# Define model names to look for
model_names = [
    f"{dataset}_{architecture}",  # Base model
    f"{dataset}_{architecture}_augmented",  # Augmented
    f"{dataset}_{architecture}_augmented_val",  # Augmented with val
]

# Add architecture-specific models
if is_image_data:
    model_names.extend([
        f"{dataset}_{architecture}_escnn",
        f"{dataset}_{architecture}_escnn_48",
    ])

if dataset == "modelnet10":
    model_names.extend([
        f"{dataset}_{architecture}_pca_then_norm_randomize_no_aug",
        f"{dataset}_{architecture}_pca_then_norm_randomize_sorted_no_aug",
        f"{dataset}_{architecture}_pointnetplus_euclidean_pca",
        f"{dataset}_{architecture2}_no_aug",
        f"{dataset}_{architecture2}_augmented",
    ])

if dataset == "tu_berlin":
    model_names.append(f"{dataset}_{architecture}_pca")

# Load results
results = {}
for model_name in model_names:
    result_path = savepath(model_name)
    if os.path.exists(result_path):
        try:
            with open(result_path, 'r') as f:
                results[model_name] = json.load(f)
        except Exception as e:
            print(f"Error loading {model_name}: {e}")

# Extract data for plotting
model_labels = []
original_acc = []
transformed_acc = []

for model_name, result in results.items():
    # Simplify label
    label = model_name.replace(f"{dataset}_{architecture}_", "").replace(f"{dataset}_", "")
    model_labels.append(label)

    # Get accuracies
    orig = result.get('original_test_results', {}).get('accuracy', 0) * 100
    trans = result.get('transformed_test_results', {}).get('accuracy', 0) * 100

    original_acc.append(orig)
    transformed_acc.append(trans)

# Create bar plot
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(model_labels))
width = 0.35

bars1 = ax.bar(x - width / 2, original_acc, width, label='Original Test', alpha=0.8)
bars2 = ax.bar(x + width / 2, transformed_acc, width, label='Transformed Test', alpha=0.8)

# Customize plot
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title(f'Model Performance Comparison - {dataset.upper()} ({transform_name})',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim([0, 105])


# Add value labels on bars
def autolabel(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height,
                f'{height:.1f}',
                ha='center', va='bottom', fontsize=8)


autolabel(bars1)
autolabel(bars2)

plt.show()

# Print summary statistics
print("\n" + "=" * 80)
print(f"RESULTS SUMMARY FOR {dataset.upper()} - {transform_name}")
print("=" * 80)
for i, label in enumerate(model_labels):
    print(
        f"{label:40s} | Original: {original_acc[i]:5.2f}% | Transformed: {transformed_acc[i]:5.2f}% | Gap: {abs(original_acc[i] - transformed_acc[i]):5.2f}%")
print("=" * 80)

In [ ]:
ds_namemap = {
    "bigger_mnist": "BigMNIST",
    "bigger_emnist": "BigEMNIST",
    "coil100": "COIL-100",
    "tu_berlin": "TU Berlin",
    "modelnet10": "ModelNet10",
}

In [ ]:
import json
import os
import math
from pathlib import Path

# Define all datasets to process
all_datasets = ["bigger_mnist", "bigger_emnist", "coil100", "modelnet10", "tu_berlin"]


def load_result(dataset_name, architecture_name, transform_name, model_suffix):
    """Load results for a specific model configuration."""
    base_path = os.path.join(
        current_path, "experiment_files", "results", dataset_name,
        architecture_name, "comparison_augmented_models", transform_name
    )

    if model_suffix is not None and "results_its" in model_suffix:
        result_path = os.path.join(base_path, model_suffix)
    else:
        model_name = f"{dataset_name}_{architecture_name}_{model_suffix}" if model_suffix else f"{dataset_name}_{architecture_name}"
        safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in model_name)
        result_path = os.path.join(base_path, f"{safe_name}.json")

    print(result_path)

    if os.path.exists(result_path):
        with open(result_path, 'r') as f:
            data = json.load(f)
            orig_acc = data.get('original_test_results', {}).get('accuracy', None)
            trans_acc = data.get('transformed_test_results', {}).get('accuracy', None)
            if orig_acc is None:
                orig_acc = data.get('original_test_results', {}).get('accuracy_mean', None)
            if trans_acc is None:
                trans_acc = data.get('transformed_test_results', {}).get('accuracy_mean', None)

            return orig_acc, trans_acc
    return None, None


def format_accuracy(acc, se=None):
    """Format accuracy values for LaTeX table, always rounding CI up."""
    if acc is None:
        return "---"
    if se is not None and se > 0:
        ci = 1.96 * se
        ci_percent = ci * 100
        ci_rounded_up = math.ceil(ci_percent * 10) / 10  # always round up to 1 decimal place
        return f"{acc * 100:.1f} $\pm$ {ci_rounded_up:.1f}"
    return f"{acc * 100:.1f}"


In [ ]:

def load_dataset_results(ds, arch):
    """Load all results for a given dataset, exposing steerable (ESCNN), PCA and ITS-best results."""
    dataset_info = get_dataset_info(ds)
    trans_name = dataset_info.transform_seq_name

    # Base and augmentation results
    orig_base, trans_base = load_result(ds, arch, trans_name, None)
    orig_aug, trans_aug = load_result(ds, arch, trans_name, "augmented")
    orig_aug_val, trans_aug_val = load_result(ds, arch, trans_name, "augmented_val")

    # Steerable (ESCNN) results for image-like datasets that used ESCNN
    escnn_datasets = ["mnist", "bigger_mnist", "emnist", "bigger_emnist", "coil100"]
    if ds in escnn_datasets:
        orig_steer, trans_steer = load_result(ds, arch, trans_name, "escnn")
    else:
        orig_steer, trans_steer = None, None

    # PCA results for tu_berlin and modelnet10 (separate column)
    if ds == "tu_berlin":
        orig_pca, trans_pca = load_result(ds, arch, trans_name, "pca")
    elif ds == "modelnet10":
        orig_pca, trans_pca = load_result(ds, arch, trans_name, "pointnetplus_euclidean_pca")
    else:
        orig_pca, trans_pca = None, None

    # Energy-based results
    orig_energy, trans_energy = load_energy_learned(dataset_info, trans_name, current_path, ds, arch)
    orig_energy_val, orig_energy_se, trans_energy_val, trans_energy_se = None, None, None, None
    if orig_energy is not None:
        orig_energy_se = orig_energy.get("accuracy_se", 0)
        trans_energy_se = trans_energy.get("accuracy_se", 0)
        orig_energy_val = orig_energy.get("accuracy_mean")
        trans_energy_val = trans_energy.get("accuracy_mean")

    # Unsupervised detector results (best)
    _, orig_unsup, trans_unsup = load_best_detector_scores_by_components(
        current_path, detectors, dataset_info, arch, ds
    )
    _, orig_unsup_se, trans_unsup_se = load_best_detector_scores_by_components(
        current_path, detectors, dataset_info, arch, ds, score_key="accuracy_se"
    )

    # ITS results (existing)
    orig_its, trans_its = load_result(ds, arch, trans_name, "results_its.json")
    orig_its_energy, trans_its_energy = load_result(ds, arch, trans_name, "results_its_energy.json")

    # ITS best detector (new column) - try common suffix used in notebook
    orig_its_best, trans_its_best = load_result(ds, arch, trans_name, "results_its_best.json")

    # Spatial Transformer Network and TTA
    orig_stn, trans_stn = load_result(ds, arch, trans_name, "spatial_transformer")
    orig_tta, trans_tta = load_result(ds, arch, trans_name, "tta")

    # Vector Neurons (ModelNet10 only)
    if ds == "modelnet10":
        orig_vn, trans_vn = load_result(ds, arch, trans_name, "vn_dgcnn")
    else:
        orig_vn, trans_vn = None, None

    return {
        'orig': {
            'base': orig_base,
            'aug': orig_aug,
            'aug_val': orig_aug_val,
            'steer': orig_steer,  # Steerable (ESCNN)
            'pca': orig_pca,  # PCA-specific result (tu_berlin / modelnet10)
            'learned': (orig_energy_val, orig_energy_se),
            'unsup': (orig_unsup, orig_unsup_se),
            'its': (orig_its, None),
            'its_energy': (orig_its_energy, None),
            'its_best': orig_its_best,  # ITS best detector
            'stn': orig_stn,
            'tta': orig_tta,
            'vn': orig_vn
        },
        'trans': {
            'base': trans_base,
            'aug': trans_aug,
            'aug_val': trans_aug_val,
            'steer': trans_steer,
            'pca': trans_pca,
            'learned': (trans_energy_val, trans_energy_se),
            'unsup': (trans_unsup, trans_unsup_se),
            'its': (trans_its, None),
            'its_energy': (trans_its_energy, None),
            'its_best': trans_its_best,
            'stn': trans_stn,
            'tta': trans_tta,
            'vn': trans_vn
        }
    }


def create_latex_table(table_type, caption, label, all_datasets):
    """Create a LaTeX table for either original or transformed test set (regular tabular)."""
    lines = [
        "\\begin{table}[htbp]",
        "\\centering",
        "\\begin{tabular}{l|cccccccccccc}",
        "\\toprule",
        ("Dataset & Base  & Augmented & Steerable (ESCNN) & PCA & Learned & Best Unsup. "
         "& ITS Energy & ITS & ITS Best & STN & TTA & VN \\\\"),
        "\\midrule"
    ]

    for ds in all_datasets:
        if ds == "coil100":
            continue
        arch = default_architecutre_mapping[ds]
        results = load_dataset_results(ds, arch)
        data = results[table_type]

        # helper to safely format entries that may be None or tuples
        def fmt(x):
            if x is None:
                return "---"
            # if x is a tuple of (mean, se) then use format_accuracy
            if isinstance(x, tuple) and len(x) == 2:
                return format_accuracy(x[0], x[1])
            return format_accuracy(x)

        row_data = [
            ds_namemap.get(ds, ds).replace("_", "\\_"),
            fmt(data['base']),
            fmt(data['aug']),
            fmt(data['steer']),
            fmt(data['pca']),
            fmt(data['learned']),
            fmt(data['unsup']),
            fmt(data['its_energy']) if data.get('its_energy') is not None and data['its_energy'][0] is not None else (
                format_accuracy(data['its'][0]) if data.get('its') is not None else "---"),
            fmt(data['its']),
            fmt(data.get('its_best')),
            fmt(data['stn']),
            fmt(data['tta']),
            fmt(data['vn'])
        ]
        lines.append(" & ".join(row_data) + " \\\\")

    lines.extend([
        "\\bottomrule",
        "\\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\end{table}"
    ])

    return lines


# Generate LaTeX tables and save to `comparison_tables.tex`
latex_lines = []
latex_lines.extend(create_latex_table('orig', 'Accuracy on Original Test Set (\\%)', 'tab:original_test', all_datasets))
latex_lines.append("")
latex_lines.extend(
    create_latex_table('trans', 'Accuracy on Transformed Test Set (\\%)', 'tab:transformed_test', all_datasets))

latex_output = "\n".join(latex_lines)
print(latex_output)

output_path = os.path.join(results_dir_path, "comparison_tables.tex")
with open(output_path, "w") as f:
    f.write(latex_output)

print(f"\n\nLaTeX tables saved to: {output_path}")